# 01. Аудит источников и проверка постановки

Этап 1 проекта «Отток абонентов СвязьРегион». Ответственный: **Алексей** (данные и репозиторий), проверка выводов: **Захар**.

Цель — описать источники, единицы наблюдения, ключи и проблемы качества, а также проверить правило активной базы и целевой переменной на реальных данных. Очистка — этап 2.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from churn.load import load_raw, parse_billing_month, contract_dates
from churn.target import active_base, make_target, campaign_size
from churn.config import RAW, CSV_KW
pd.set_option('display.width', 140)
subs, billing, events = load_raw()
preview = pd.read_csv(RAW / 'billing_preview.csv', **CSV_KW)
{n: df.shape for n, df in [('subscribers', subs), ('billing', billing), ('events', events), ('preview', preview)]}

{'subscribers': (2600, 5),
 'billing': (24248, 14),
 'events': (3299, 4),
 'preview': (60, 14)}

## 1. Единицы наблюдения и ключи

In [2]:
keys = pd.DataFrame({
    'источник': ['subscribers', 'billing_monthly', 'contract_events'],
    'единица строки': ['абонент', 'абонент × расчётный месяц', 'событие договора'],
    'ключ': ['customer_id', 'customer_id + billing_month', 'event_id'],
    'строк': [len(subs), len(billing), len(events)],
    'дубликатов ключа': [subs.customer_id.duplicated().sum(),
                         billing.duplicated(['customer_id', 'billing_month']).sum(),
                         events.event_id.duplicated().sum()],
})
keys

,источник,единица строки,ключ,строк,дубликатов ключа
0,subscribers,абонент,customer_id,2600,0
1,billing_monthly,абонент × расчётный месяц,customer_id + billing_month,24248,150
2,contract_events,событие договора,event_id,3299,0


In [3]:
print('Полных дублей строк в биллинге:', billing.duplicated().sum())
print('Preview целиком входит в основной биллинг:', len(preview.merge(billing.drop_duplicates())) == len(preview))
print('customer_id биллинга/событий вне справочника:', len(set(billing.customer_id) - set(subs.customer_id)), len(set(events.customer_id) - set(subs.customer_id)))
print('Абонентов без строк биллинга:', len(set(subs.customer_id) - set(billing.customer_id)))

Полных дублей строк в биллинге: 150
Preview целиком входит в основной биллинг: True
customer_id биллинга/событий вне справочника: 0 0
Абонентов без строк биллинга: 50


## 2. Проблемы качества (фиксируем, исправляем на этапе 2)

In [4]:
bm_bad = ~billing.billing_month.str.match(r'^\d{4}-\d{2}-01$')
paid_comma = billing.paid_rub.str.contains(',', na=False)
voice = pd.to_numeric(billing.voice_minutes)
issues = pd.Series({
    'billing: полные дубли строк': billing.duplicated().sum(),
    'billing: billing_month в формате DD.MM.YYYY': bm_bad.sum(),
    'billing: tariff нестандартное написание': (~billing.tariff.isin(['Basic', 'Comfort', 'Premium'])).sum(),
    'billing: paid_rub с десятичной запятой': paid_comma.sum(),
    'billing: closing_debt_rub пропуск': billing.closing_debt_rub.isna().sum(),
    'billing: traffic_gb пропуск': billing.traffic_gb.isna().sum(),
    'billing: voice_minutes < 0': (voice < 0).sum(),
    'subscribers: activation_date пропуск': subs.activation_date.isna().sum(),
})
issues.to_frame('строк')

,строк
billing: полные дубли строк,150
billing: billing_month в формате DD.MM.YYYY,400
billing: tariff нестандартное написание,200
billing: paid_rub с десятичной запятой,120
billing: closing_debt_rub пропуск,40
billing: traffic_gb пропуск,240
billing: voice_minutes < 0,80
subscribers: activation_date пропуск,65


In [5]:
billing.tariff.value_counts()

tariff
Comfort    9641
Basic      9528
Premium    4879
COMFORT      93
 basic       77
Премиум      30
Name: count, dtype: int64

Проверка тождества долга `closing = opening + charged − paid` на строках без пропусков:

In [6]:
b = billing.drop_duplicates().copy()
for c in ['charged_rub', 'paid_rub', 'opening_debt_rub', 'closing_debt_rub']:
    b[c] = pd.to_numeric(b[c].str.replace(',', '.'))
diff = (b.opening_debt_rub + b.charged_rub - b.paid_rub - b.closing_debt_rub).abs()
print('нарушений > 0.01 руб.:', (diff > 0.01).sum(), '| пропусков closing (восстановимы по тождеству):', b.closing_debt_rub.isna().sum())
b['bm'] = parse_billing_month(b.billing_month)
print('ключ после приведения формата месяца уникален:', not b.duplicated(['customer_id', 'bm']).any())
print('available_at = billing_month + 1 месяц везде:', (pd.to_datetime(b.available_at) == b.bm + pd.offsets.MonthBegin(1)).all())

нарушений > 0.01 руб.: 0 | пропусков closing (восстановимы по тождеству): 40
ключ после приведения формата месяца уникален: True
available_at = billing_month + 1 месяц везде: True


## 3. Согласованность справочника и журнала событий

In [7]:
contracts = contract_dates(events)
sa = pd.to_datetime(subs.set_index('customer_id').activation_date)
known = sa.dropna()
print('activation_date справочника совпадает с событием activation:', (known == contracts.loc[known.index, 'activation']).all())
print('пропущенных activation_date восстанавливается из событий:', contracts.loc[sa[sa.isna()].index, 'activation'].notna().sum())
print(events.event_type.value_counts().to_dict())
print('подключения:', contracts.activation.min().date(), '—', contracts.activation.max().date())
print('расторжения:', contracts.termination.min().date(), '—', contracts.termination.max().date())

activation_date справочника совпадает с событием activation: True
пропущенных activation_date восстанавливается из событий: 65
{'activation': 2600, 'termination': 699}
подключения: 2019-01-02 — 2026-09-29
расторжения: 2025-09-03 — 2026-09-30


## 4. Активная база и доля оттока по срезам

Дата расчёта t — 1-е число месяца, горизонт — календарный месяц `[t, t+1 мес.)`.

In [8]:
rows = []
for t in pd.date_range('2025-10-01', '2026-10-01', freq='MS'):
    y = make_target(contracts, t)
    n = len(y)
    rows.append({'дата расчёта': t.date(), 'активная база': n,
                 'отток за горизонт': y.churn.sum() if y.churn.notna().all() else None,
                 'доля оттока, %': round(y.churn.mean() * 100, 2) if y.churn.notna().all() else None,
                 'лимит кампании (10%)': campaign_size(n)})
snap = pd.DataFrame(rows)
snap

,дата расчёта,активная база,отток за горизонт,"доля оттока, %",лимит кампании (10%)
0,2025-10-01,1958,43.0,2.20,195
1,2025-11-01,1965,46.0,2.34,196
2,2025-12-01,1969,48.0,2.44,196
3,2026-01-01,1971,50.0,2.54,197
4,2026-02-01,1971,52.0,2.64,197
5,2026-03-01,1969,54.0,2.74,196
6,2026-04-01,1965,56.0,2.85,196
7,2026-05-01,1959,58.0,2.96,195
8,2026-06-01,1951,60.0,3.08,195
9,2026-07-01,1941,62.0,3.19,194


Доля оттока растёт с 2,2 % до 3,4 % в месяц. На 1 сентября 2026: база 1916, ушло 65 (3,39 %), лимит кампании 191 — даже идеальная модель может покрыть все 65 случаев в пределах лимита. Для 1 октября 2026 исход не наблюдается: это боевой срез.

## 5. Длина истории на дату расчёта

In [9]:
t = pd.Timestamp('2026-09-01')
base = active_base(contracts, t)
known_b = b[pd.to_datetime(b.available_at) <= t]
hist = known_b.groupby('customer_id').size().reindex(base).fillna(0).astype(int)
tenure_m = (t - contracts.loc[base, 'activation']).dt.days / 30.44
print('месяцев истории биллинга у активной базы на 2026-09-01:')
print(hist.value_counts().sort_index().to_dict())
print('стаж < 3 мес.:', (tenure_m < 3).sum(), '| стаж < 6 мес.:', (tenure_m < 6).sum())

месяцев истории биллинга у активной базы на 2026-09-01:
{1: 50, 2: 49, 3: 45, 4: 43, 5: 40, 6: 41, 7: 38, 8: 41, 9: 39, 10: 36, 11: 31, 12: 1463}
стаж < 3 мес.: 144 | стаж < 6 мес.: 268


История ограничена 12 месяцами выгрузки, у новых абонентов — 1–11 месяцев. Отсутствие ранних месяцев кодируем как пропуск + признак `n_history_months`, а не нулём.